In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow import keras

from art.attacks.evasion import FastGradientMethod
from art.estimators.classification import TensorFlowV2Classifier

In [27]:
(_, _), (x_test, y_test) = cifar10.load_data()
x_test = x_test.astype("float32") / 255.0
y_test = y_test.flatten()

In [ ]:
model = keras.models.load_model("model.keras")

classifier = TensorFlowV2Classifier(
    model=model,
    loss_object=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    nb_classes=10,
    input_shape=(32, 32, 3),
    clip_values=(0, 1),
)

In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

def fgsm_attack(model, images, labels, eps=8/255):
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.int32)

    with tf.GradientTape() as tape:
        tape.watch(images)
        predictions = model(images, training=False)
        loss = loss_fn(labels, predictions)

    grad       = tape.gradient(loss, images)
    adv_images = images + eps * tf.sign(grad)
    adv_images = tf.clip_by_value(adv_images, 0.0, 1.0)
    return adv_images

In [31]:
EPS = 8 / 255
N   = 1000

x_sample = x_test[:N]
y_sample  = y_test[:N]

# --- Baseline: czysta dokładność ---
clean_preds = np.argmax(classifier.predict(x_sample), axis=1)
clean_acc   = np.mean(clean_preds == y_sample)

# --- ART FGSM ---
art_attack  = FastGradientMethod(estimator=classifier, eps=EPS)
x_adv_art   = art_attack.generate(x=x_sample)
art_preds   = np.argmax(classifier.predict(x_adv_art), axis=1)
art_acc     = np.mean(art_preds == y_sample)

# --- Własna implementacja FGSM ---
x_adv_own  = fgsm_attack(model, x_sample, y_sample, eps=EPS).numpy()
own_preds  = np.argmax(classifier.predict(x_adv_own), axis=1)
own_acc    = np.mean(own_preds == y_sample)

print(f"Clean accuracy      : {clean_acc:.4f}")
print(f"ART FGSM accuracy   : {art_acc:.4f}  (eps={EPS:.4f})")
print(f"Own FGSM accuracy   : {own_acc:.4f}  (eps={EPS:.4f})")

NameError: name 'classifier' is not defined